# Lab 5 - Validate results
Checks the tables after a pipeline update. Takes a consistent snapshot of every table, checks that none are empty, that Netflix Silver has no duplicate `show_id` and that neither Silver table breaks a rule. Every Kafka delivery must land in exactly one place: Silver or quarantine.
Silver and quarantine are recomputed from Bronze with the shared code and must match the pipeline output. Repeated `event_id`s at different offsets are only counted.

In [ ]:
%run ./lab5_00_config

In [ ]:
names = ["titles_bronze", "titles_silver", "wiki_bronze", "wiki_silver", "wiki_quarantine"]
stream_names = ["wiki_bronze", "wiki_silver", "wiki_quarantine"]
versions = {name: delta_version(name) for name in stream_names}
frames = {name: read_version(name, version) for name, version in versions.items()}
# Materialized views do not support time travel. Copy their current result into ordinary Delta snapshot tables.
for name in ["titles_bronze", "titles_silver"]:
    snapshot_name = table("validation_snapshot_" + name)
    spark.table(table(name)).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(snapshot_name)
    frames[name] = spark.table(snapshot_name)
counts = {name: df.count() for name, df in frames.items()}
print({"versions": versions, "counts": counts})
assert counts["titles_bronze"] == 8807
assert counts["titles_silver"] > 0
assert counts["wiki_bronze"] > 0
assert counts["wiki_silver"] > 0
assert frames["titles_silver"].groupBy("show_id").count().filter("count > 1").limit(1).count() == 0

keys = ["kafka_topic", "kafka_partition", "kafka_offset"]
assert frames["wiki_bronze"].groupBy(*keys).count().filter("count > 1").limit(1).count() == 0
routed = frames["wiki_silver"].select(*keys).unionByName(frames["wiki_quarantine"].select(*keys))
assert_equal(routed, frames["wiki_bronze"], keys, "every raw delivery routed exactly once")
assert counts["wiki_silver"] + counts["wiki_quarantine"] == counts["wiki_bronze"]

assert annotate_quality(frames["titles_silver"], TITLE_RULES).filter("NOT _is_valid").limit(1).count() == 0
assert annotate_quality(frames["wiki_silver"], WIKI_RULES).filter("NOT _is_valid").limit(1).count() == 0
prepared = annotate_quality(parse_wiki(frames["wiki_bronze"]), WIKI_RULES)
assert_equal(frames["wiki_quarantine"], prepared.filter("NOT _is_valid"), keys + ["_quality_errors"], "quarantine reasons")
assert_equal(frames["titles_silver"], valid_rows(clean_titles(frames["titles_bronze"]), TITLE_RULES), TITLE_BUSINESS_COLUMNS, "Netflix Silver")
assert_equal(frames["wiki_silver"], prepared.filter("_is_valid"), WIKI_BUSINESS_COLUMNS + keys, "Wikipedia Silver")

business_duplicates = frames["wiki_silver"].groupBy("event_id").count().filter("count > 1").count()
print("Business event IDs repeated across distinct Kafka deliveries:", business_duplicates)
print("These are upstream repeats, not repeated checkpoint ingestion; no business deduplication is claimed.")
assert versions == {name: delta_version(name) for name in stream_names}, "Pipeline changed during validation; retry after it stops"
for name in ["titles_bronze", "titles_silver"]:
    assert_equal(spark.table(table(name)), frames[name], frames[name].columns, f"{name} unchanged during validation")
print("PASS: all final validation checks")